# GraphSAGE for Collusion-Ring Detection

A cell-by-cell walkthrough of `graphsage_collusion.py`, the relational half of the hybrid XGBoost + GraphSAGE fraud detector. The code is the same as the script, split into stages so each step can be inspected on its own.

**Why a GNN?** XGBoost scores one claim row at a time. It catches cost-anomaly fraud (upcoding, phantom billing) well, but a collusion ring leaves almost no tabular trace. Its only real anomaly is relational: a small set of providers and patients transacting far more densely than geography would explain. GraphSAGE builds each node's embedding from its neighbours' embeddings, so a ring provider ends up looking like "connected to a tight, repeating cluster of patients".

**Task:** edge classification. Each claim is an edge in a bipartite patient-provider graph, and the model predicts whether that claim belongs to the `collusion_ring` fraud type.

**Requirements:** `torch`, `torch_geometric`, `scikit-learn`, `pandas`, `numpy`. Run from the project root, where `output/kenyanised/master_claims_sample_500k_13pct_v2.csv` lives.

## 1. Imports and configuration

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, f1_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

t0 = time.time()
torch.manual_seed(42)
np.random.seed(42)

RANDOM_STATE = 42
DATA_PATH = "output/kenyanised/master_claims_sample_500k_13pct_v2.csv"
OUTPUT_DIR = "output/model"

## 2. Load the claims
Only the columns the graph needs: the two node keys, the node attributes, and the labels. Labels are used for the loss and evaluation, never as node features.

In [ ]:
print("Loading data...")
df = pd.read_csv(DATA_PATH, usecols=[
    "PATIENT_ID", "PROVIDER_ID", "GENDER", "AGE", "PATIENT_COUNTY",
    "PROVIDER_COUNTY", "PROVIDER_LEVEL", "IS_FRAUD", "FRAUD_TYPE",
])
print(f"Loaded {len(df)} claims")

## 3. Reproduce the XGBoost train/val/test split

The split is identical to `xgboost_model_v2_robust.py` (`RANDOM_STATE=42`, 70/15/15, stratified on `IS_FRAUD`). That makes the test-set `collusion_ring` rows the same 3,246 claims the XGBoost recall number was measured on, so the comparison is fair and the two models can be combined row-for-row in the hybrid.

In [ ]:
y_full = df["IS_FRAUD"]
idx_train, idx_temp = train_test_split(
    df.index, test_size=0.30, random_state=RANDOM_STATE, stratify=y_full
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_full.loc[idx_temp]
)
split = pd.Series("train", index=df.index)
split.loc[idx_val] = "val"
split.loc[idx_test] = "test"
df["SPLIT"] = split.values
print(df["SPLIT"].value_counts())

n_collusion_test = ((df["SPLIT"] == "test") & (df["FRAUD_TYPE"] == "collusion_ring")).sum()
print(f"collusion_ring rows in test split: {n_collusion_test} "
      f"(should match the 3246 from the XGBoost acid test)")

## 4. Build the node ID space

Patients get IDs first, providers after. Each claim becomes an edge between its patient node and its provider node.

In [ ]:
patient_ids = df["PATIENT_ID"].unique()
provider_ids = df["PROVIDER_ID"].unique()
n_patients = len(patient_ids)
n_providers = len(provider_ids)
n_nodes = n_patients + n_providers
print(f"\nGraph: {n_patients} patient nodes + {n_providers} provider nodes = {n_nodes} nodes, {len(df)} claim edges")

patient_node_id = {pid: i for i, pid in enumerate(patient_ids)}
provider_node_id = {pid: n_patients + i for i, pid in enumerate(provider_ids)}

df["PATIENT_NODE"] = df["PATIENT_ID"].map(patient_node_id)
df["PROVIDER_NODE"] = df["PROVIDER_ID"].map(provider_node_id)

## 5. Node features

Features are deliberately non-cost, so the GNN complements XGBoost instead of repeating it. Layout: `[is_patient, is_provider, age_norm, gender_M, gender_F, county one-hot, provider-level one-hot]`, zero-padded for the node type a field doesn't apply to.

**Node degree is excluded on purpose.** `degree_only_baseline.py` showed that two features (log patient degree and log provider degree) separated the ring perfectly, because the original injector reused a small fixed pool of ring patients. That is a counting shortcut, not graph reasoning. The injector was fixed (60 → 300 ring patients), and degree stays out of the features so the model must use neighbourhood composition.

In [ ]:
counties = sorted(set(df["PATIENT_COUNTY"].astype(str).unique()) | set(df["PROVIDER_COUNTY"].astype(str).unique()))
county_idx = {c: i for i, c in enumerate(counties)}
levels = sorted(df["PROVIDER_LEVEL"].astype(str).unique())
level_idx = {l: i for i, l in enumerate(levels)}
n_county = len(counties)
n_level = len(levels)

# feature layout: [is_patient, is_provider, age_norm, gender_M, gender_F,
#                   county(n_county), level(n_level)]
# NOTE: raw/log node degree was DELIBERATELY REMOVED from the feature set.
# degree_only_baseline.py proved a 2-feature logistic regression on just
# log(patient_degree)/log(provider_degree) alone got PERFECT separation
# (PR-AUC 1.0) on collusion_ring - because the injector reused a small
# fixed ring-patient pool, total appearance count was a trivial tabular
# "tell" with no graph reasoning involved at all, and GraphSAGE had direct
# access to that same shortcut through this feature. The injector fix
# (N_RING_PATIENTS 60->300, see inject_collusion_ring.py) closes the worst
# of this at the DATA level, but keeping degree out of the node features
# entirely closes it at the MODEL level too, forcing GraphSAGE to rely on
# actual neighbourhood composition (which specific neighbours a node has,
# aggregated through SAGEConv's mean operator) rather than a magnitude
# count that was never a genuinely relational signal.
feat_dim = 2 + 1 + 2 + n_county + n_level
X = np.zeros((n_nodes, feat_dim), dtype=np.float32)
print(f"\nNode feature dim: {feat_dim} (counties={n_county}, levels={n_level}) - degree EXCLUDED")

# patient-node features: first occurrence per patient
patient_first = df.drop_duplicates("PATIENT_ID", keep="first").set_index("PATIENT_ID")
age_mean, age_std = patient_first["AGE"].mean(), patient_first["AGE"].std()
for pid, node in patient_node_id.items():
    row = patient_first.loc[pid]
    X[node, 0] = 1.0
    X[node, 2] = (row["AGE"] - age_mean) / (age_std if age_std else 1.0)
    if str(row["GENDER"]) == "M":
        X[node, 3] = 1.0
    else:
        X[node, 4] = 1.0
    X[node, 5 + county_idx[str(row["PATIENT_COUNTY"])]] = 1.0

# provider-node features: first occurrence per provider
provider_first = df.drop_duplicates("PROVIDER_ID", keep="first").set_index("PROVIDER_ID")
for pid, node in provider_node_id.items():
    row = provider_first.loc[pid]
    X[node, 1] = 1.0
    X[node, 5 + county_idx[str(row["PROVIDER_COUNTY"])]] = 1.0
    X[node, 5 + n_county + level_idx[str(row["PROVIDER_LEVEL"])]] = 1.0

print(f"Node feature build done in {time.time()-t0:.1f}s")

## 6. Edge index and tensors

The graph is **transductive**: message passing uses every claim edge (train, val and test), because who treated whom is observable at inference time. Only the fraud labels of val/test edges are hidden from the loss. The label is still predicted from claims the model never trained on, and only the existence of an edge is used for structure.

In [ ]:
src = df["PATIENT_NODE"].to_numpy()
dst = df["PROVIDER_NODE"].to_numpy()
edge_index = np.stack([
    np.concatenate([src, dst]),
    np.concatenate([dst, src]),
])

x_t = torch.tensor(X, dtype=torch.float32)
edge_index_t = torch.tensor(edge_index, dtype=torch.long)

# edge-level (claim-level) labels + masks, in ORIGINAL row order
y_ring = (df["FRAUD_TYPE"] == "collusion_ring").astype(np.float32).to_numpy()
edge_src_t = torch.tensor(src, dtype=torch.long)
edge_dst_t = torch.tensor(dst, dtype=torch.long)
y_t = torch.tensor(y_ring, dtype=torch.float32)

train_mask = torch.tensor((df["SPLIT"] == "train").to_numpy())
val_mask = torch.tensor((df["SPLIT"] == "val").to_numpy())
test_mask = torch.tensor((df["SPLIT"] == "test").to_numpy())
fraud_type_arr = df["FRAUD_TYPE"].to_numpy()

print(f"Tensors built. edge_index shape {edge_index_t.shape}, x shape {x_t.shape}")

## 7. Model: 2-layer GraphSAGE encoder + MLP edge head

Two `SAGEConv` layers (mean aggregation) produce a 32-dimensional embedding per node. For a claim, the patient and provider embeddings are concatenated and passed through a small MLP that outputs a ring logit.

In [ ]:
class GraphSAGEEdgeClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=32):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.edge_mlp = nn.Sequential(
            nn.Linear(out_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def encode(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.2, training=self.training)
        h = self.conv2(h, edge_index)
        return h

    def edge_logits(self, h, edge_src, edge_dst):
        e = torch.cat([h[edge_src], h[edge_dst]], dim=1)
        return self.edge_mlp(e).squeeze(-1)

## 8. Optimiser and class imbalance

Ring claims are rare, so the loss uses `pos_weight = n_negative / n_positive` computed on the training edges only.

In [ ]:
device = torch.device("cpu")
model = GraphSAGEEdgeClassifier(in_dim=feat_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

n_pos = y_t[train_mask].sum().item()
n_neg = train_mask.sum().item() - n_pos
pos_weight = torch.tensor(n_neg / max(n_pos, 1))
print(f"\nTrain edges: {int(train_mask.sum())}  pos={int(n_pos)} neg={int(n_neg)}  pos_weight={pos_weight:.2f}")
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## 9. Train

Full-batch training, 60 epochs. The checkpoint with the best **validation** PR-AUC is kept, so the test set plays no part in model selection.

In [ ]:
CHECKPOINT_PATH = f"{OUTPUT_DIR}/graphsage_collusion_checkpoint.pt"
N_EPOCHS = 60
best_val_prauc = -1.0
best_state = None
history = []

print(f"\nTraining for up to {N_EPOCHS} epochs (full-batch, transductive)...")
t_train0 = time.time()
for epoch in range(1, N_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    h = model.encode(x_t, edge_index_t)
    logits = model.edge_logits(h, edge_src_t, edge_dst_t)
    loss = loss_fn(logits[train_mask], y_t[train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        h = model.encode(x_t, edge_index_t)
        logits = model.edge_logits(h, edge_src_t, edge_dst_t)
        val_probs = torch.sigmoid(logits[val_mask]).numpy()
        val_y = y_t[val_mask].numpy()
        val_prauc = average_precision_score(val_y, val_probs)

    history.append({"epoch": epoch, "train_loss": loss.item(), "val_pr_auc": val_prauc})
    if val_prauc > best_val_prauc:
        best_val_prauc = val_prauc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        print(f"  epoch {epoch:3d}  train_loss={loss.item():.4f}  val_pr_auc={val_prauc:.4f}  "
              f"elapsed={time.time()-t_train0:.1f}s")

print(f"\nTraining done in {time.time()-t_train0:.1f}s. Best val PR-AUC: {best_val_prauc:.4f}")
model.load_state_dict(best_state)
torch.save({"model_state": best_state, "feat_dim": feat_dim}, CHECKPOINT_PATH)
pd.DataFrame(history).to_csv(f"{OUTPUT_DIR}/graphsage_training_history.csv", index=False)

### Training curve

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(h["epoch"], h["train_loss"]); ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(h["epoch"], h["val_pr_auc"]); ax[1].set_title("Validation PR-AUC"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 10. Evaluate on the test set
Scores are ring-vs-everything-else at the default 0.5 cut-off. The false-positive-style check shows how often ordinary, upcoded and phantom-billed claims are wrongly called ring claims.

In [ ]:
model.eval()
with torch.no_grad():
    h = model.encode(x_t, edge_index_t)
    logits = model.edge_logits(h, edge_src_t, edge_dst_t)
    probs = torch.sigmoid(logits).numpy()

test_mask_np = test_mask.numpy()
test_probs = probs[test_mask_np]
test_y = y_ring[test_mask_np]
test_fraud_type = fraud_type_arr[test_mask_np]

test_pred = (test_probs >= 0.5).astype(int)
pr_auc = average_precision_score(test_y, test_probs)
f1 = f1_score(test_y, test_pred)
print(f"\n{'='*60}\nTEST SET RESULTS (GraphSAGE, collusion_ring vs everything else)\n{'='*60}")
print(f"PR-AUC: {pr_auc:.4f}   F1: {f1:.4f}")

ring_mask = test_fraud_type == "collusion_ring"
ring_recall = recall_score(test_y[ring_mask], test_pred[ring_mask]) if ring_mask.sum() else float("nan")
print(f"\ncollusion_ring recall (GraphSAGE): n={ring_mask.sum()} "
      f"caught={int(test_pred[ring_mask].sum())} recall={ring_recall:.4f} "
      f"mean_score={test_probs[ring_mask].mean():.4f}")
print("(compare to XGBoost's 54.0% recall on the SAME 3246 test rows, with no graph structure)")

for ft in ["none", "upcoding", "phantom_billing"]:
    m = test_fraud_type == ft
    if m.sum():
        print(f"  false-positive-style check on '{ft}': n={m.sum()} "
              f"flagged_as_ring={int(test_pred[m].sum())} ({test_pred[m].mean()*100:.2f}%) "
              f"mean_score={test_probs[m].mean():.4f}")

In [ ]:
results_df = pd.DataFrame({
    "FRAUD_TYPE": test_fraud_type,
    "y_true_ring": test_y,
    "pred_ring": test_pred,
    "score": test_probs,
})
results_df.to_csv(f"{OUTPUT_DIR}/graphsage_test_predictions.csv", index=False)
print(f"\nSaved checkpoint to {CHECKPOINT_PATH}")
print(f"Saved test predictions to {OUTPUT_DIR}/graphsage_test_predictions.csv")
print(f"TOTAL elapsed: {time.time()-t0:.1f}s")

## 11. Ablation: topology only

`graphsage_topology_only.py` is the same model with node features reduced to the type indicator, with no age, gender, county or level. `attrs_only_baseline.py` showed that demographics alone carry some signal, so this ablation asks how much of GraphSAGE's result is connectivity rather than attributes. Run that script and compare its history below.

In [ ]:
import os
hp = f"{OUTPUT_DIR}/graphsage_topology_only_history.csv"
if os.path.exists(hp):
    t = pd.read_csv(hp)
    print("Full features   - best val PR-AUC:", round(h["val_pr_auc"].max(), 4))
    print("Topology only   - best val PR-AUC:", round(t["val_pr_auc"].max(), 4))
else:
    print("Run graphsage_topology_only.py first.")

## 12. How GraphSAGE fits into the hybrid

The deployed detector flags a claim if **either** model fires: `XGBoost score >= t_xgb` **or** `GraphSAGE ring score >= t_graphsage`. The thresholds are tuned on the validation split only (`tune_hybrid_threshold.py`), then applied once to test.

In [ ]:
print(open(f"{OUTPUT_DIR}/hybrid_tuned_thresholds.txt").read())
pd.read_csv(f"{OUTPUT_DIR}/per_fraudtype_recall_v2data.csv")

### Robustness of the hybrid
Results from `hybrid_robustness_test.py`. Note the GraphSAGE-specific conditions (3 and 4) and the combined worst case (5).

In [ ]:
r = pd.read_csv(f"{OUTPUT_DIR}/hybrid_robustness_results.csv")
r[["condition", "xgboost_alone_recall", "hybrid_tuned_recall", "hybrid_tuned_precision",
   "recall_collusion_ring_xgb", "recall_collusion_ring_hybrid"]].round(3)

## 13. Limitations to state in the write-up

- The ring was **injected synthetically** (`inject_collusion_ring.py`), so GraphSAGE's recall on it measures detection of that injected pattern, not of real-world collusion.
- The transductive setup needs the graph to be known at scoring time. New patients or providers with no history weaken the signal, which the "missing history" robustness test shows (ring recall falls from 100% to about 76% at 25% missing history).
- In the combined worst case the hybrid's advantage largely disappears, so the two models should be monitored together.
- Thresholds were tuned on validation data only, but the data are still synthetic, so deployment would need re-tuning on real claims.